# Los Angeles restaurant market
## Research for a robot-served café concept

**Decision brief for investors** · October 2026

This investor analysis tests the market context for a robot-served café: supply mix, chain structure, café-chain comparables, seat capacity and street concentration. The inventory does not measure demand, profitability, customer acceptance or robot payback; those require local diligence and a measured operating pilot.


**#### PDF Presentation**

Static site presentation: <[https://la-restaurant-market-analysis.onrender.com](https://la-restaurant-market-analysis.onrender.com)>
Documentation for analysis: <[https://github.com/carolacezalves/la-restaurant-market-analysis/tree/main](https://github.com/carolacezalves/la-restaurant-market-analysis/tree/main)>


### Executive lens
The dataset is a cross-sectional inventory, not a time series or customer-demand sample. Treat findings as market context. The three capacity scenarios are options for learning, not a final seat-count recommendation.
### Analysis guide
1. Data quality and market mix
2. Chain structure and capacity
3. Café-chain comparison
4. Simpson’s paradox check
5. Street patterns
6. Search-interest diligence with Wordstat
7. Investor decision and pilot scenarios

## 1. Data loading and quality controls

Source file: /datasets/rest_data_us.csv. If running outside the original course environment, update DATA_PATH to the supplied CSV. We retain a raw copy, normalize column names and types, and report missingness, duplicate identifiers, invalid seat counts, and parsing decisions before analysis.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

DATA_PATH = Path('/datasets/rest_data_us.csv')
if not DATA_PATH.exists():
    # Search beside the notebook and in the repository's standard data folder.
    candidates = [
        Path.cwd() / 'rest_data_us.csv',
        Path.cwd() / 'data' / 'rest_data_us.csv',
        Path.cwd().parent / 'rest_data_us.csv',
        Path.cwd().parent / 'data' / 'rest_data_us.csv',
    ]
    DATA_PATH = next((path for path in candidates if path.exists()), None)
    if DATA_PATH is None:
        raise FileNotFoundError('Put rest_data_us.csv beside the notebook or in the repository data/ folder, then update DATA_PATH.')

raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(raw):,} rows × {raw.shape[1]} columns from {DATA_PATH}')
display(raw.head())
print('Columns:', raw.columns.tolist())

In [ ]:
# Standardize headers and parse fields into explicit analytical types.
df = raw.copy()
df.columns = df.columns.str.strip().str.lower()
expected = {'id', 'object_name', 'address', 'chain', 'object_type', 'number'}
missing_columns = expected - set(df.columns)
if missing_columns:
    raise ValueError(f'Required columns missing: {sorted(missing_columns)}')

df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')
df['number'] = pd.to_numeric(df['number'], errors='coerce')
df['object_name'] = df['object_name'].astype('string').str.strip()
df['address'] = df['address'].astype('string').str.strip()
df['object_type'] = df['object_type'].astype('string').str.strip().str.title()
chain_map = {'true': True, 'false': False, '1': True, '0': False}
df['chain'] = df['chain'].astype('string').str.strip().str.lower().map(chain_map).astype('boolean')

quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(dropna=True)
})
display(quality)
print(f'Duplicate IDs: {df.id.duplicated().sum():,}')
print(f'Duplicate full rows: {df.duplicated().sum():,}')
print(f'Invalid / nonpositive seat counts: {(df.number.isna() | (df.number <= 0)).sum():,}')
print(f'Unknown chain values after normalization: {df.chain.isna().sum():,}')

### Cleaning decisions

The chain field is a market classification. Missing classifications are not silently assigned to independent businesses; they are excluded from chain-share comparisons and counted separately. Rows with missing type or invalid seat count are excluded only from analyses that require those fields. Duplicate IDs are investigated and removed only if the source contains actual repeated records. These rules preserve the distinction between a true independent venue and an unknown status.

In [ ]:
# Resolve duplicate IDs conservatively: retain the first exact ID record and report the count.
duplicate_id_rows = int(df.id.duplicated(keep='first').sum())
df = df.drop_duplicates(subset='id', keep='first').copy()

# Keep rows with unknown chain status for general venue analysis.
valid_type = df[df.object_type.notna() & df.object_type.ne('')].copy()
valid_seats = valid_type[valid_type.number.notna() & df.number.gt(0)].copy()
classified = valid_type[valid_type.chain.notna()].copy()

print(f'Analysis rows after ID de-duplication: {len(df):,}')
print(f'Valid venue types: {len(valid_type):,}; valid seat counts: {len(valid_seats):,}')
print(f'Known chain classification: {len(classified):,}; unknown: {df.chain.isna().sum():,}')
print(f'Duplicate ID rows removed: {duplicate_id_rows:,}')

## 2. Market composition

### Venue types
The first chart shows the mix of venue categories. This is a composition view: a large restaurant category means a larger count in this file, not necessarily higher revenue or unmet demand.

In [ ]:
sns.set_theme(style='whitegrid', palette='deep', context='notebook')
type_counts = valid_type.object_type.value_counts().rename_axis('type').reset_index(name='venues')
type_counts['share_pct'] = 100 * type_counts.venues / type_counts.venues.sum()
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=type_counts, y='type', x='venues', color='#197C78', ax=ax)
ax.set(title='Los Angeles venues by establishment type', xlabel='Number of venues', ylabel='')
for i, r in type_counts.iterrows(): ax.text(r.venues, i, f'  {r.venues:,} · {r.share_pct:.1f}%', va='center', fontsize=9)
sns.despine(); plt.tight_layout(); plt.show()
display(type_counts)

In [ ]:
top_type = type_counts.iloc[0]
display(Markdown(f"**Interpretation.** {top_type['type']} is the largest recorded category ({top_type.venues:,}, {top_type.share_pct:.1f}% of classified venues). This establishes the competitive baseline, but does not identify the best segment for a robot-served café; differentiation and customer economics are not measured here."))

### Chain versus independent venues

Unknown chain labels are excluded from the denominator and disclosed. We compare both location counts and type mix to see where chains are most represented.

In [ ]:
chain_counts = classified.chain.value_counts().rename(index={True:'Chain', False:'Independent'}).rename_axis('status').reset_index(name='venues')
chain_counts['share_pct'] = chain_counts.venues / chain_counts.venues.sum() * 100
fig, ax = plt.subplots(figsize=(7, 4.6))
sns.barplot(data=chain_counts, x='status', y='share_pct', palette=['#D98841','#197C78'], ax=ax)
ax.set(title='Share of venues by chain status', xlabel='', ylabel='Share of classified venues (%)')
for i,r in chain_counts.iterrows(): ax.text(i, r.share_pct, f'{r.share_pct:.1f}%\n(n={r.venues:,})', ha='center', va='bottom')
ax.set_ylim(0, chain_counts.share_pct.max()*1.2); sns.despine(); plt.tight_layout(); plt.show()
display(chain_counts)

In [ ]:
type_chain = (classified.groupby(['object_type','chain']).size().rename('venues').reset_index())
type_chain['share_within_type_pct'] = type_chain['venues'] / type_chain.groupby('object_type').venues.transform('sum') * 100
fig, ax = plt.subplots(figsize=(10,5.5))
sns.barplot(data=type_chain, y='object_type', x='share_within_type_pct', hue='chain', palette={False:'#197C78',True:'#D98841'}, ax=ax)
ax.set(title='Chain share within each venue type', xlabel='Share of type with known chain status (%)', ylabel='')
ax.legend(title='Chain', labels=['Independent','Chain']); sns.despine(); plt.tight_layout(); plt.show()
display(type_chain.pivot(index='object_type',columns='chain',values='share_within_type_pct').rename(columns={False:'Independent %',True:'Chain %'}).round(1))

In [ ]:
chain_by_type = type_chain[type_chain.chain].sort_values('venues', ascending=False)
most_common_chain_type = chain_by_type.iloc[0]
display(Markdown(f"**Interpretation.** By absolute number of chain locations, {most_common_chain_type.object_type} is the most common chain type ({most_common_chain_type.venues:,}). Read the within-type percentages alongside counts: absolute prevalence and propensity to be a chain answer different questions. Missing chain labels ({df.chain.isna().sum():,}) are not treated as independent."))

## 3. Seating capacity and chain structure

Means are paired with medians and quartiles because venue capacity is usually skewed. The chain comparison uses distributions as well as averages: many small sites and a few large sites can produce similar means.

In [ ]:
seat_summary = (valid_seats.groupby('object_type').number
    .agg(venues='count', mean='mean', median='median', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75))
    .sort_values('mean', ascending=False).round(1))
display(seat_summary)
fig, ax = plt.subplots(figsize=(10,5.5))
sns.barplot(data=valid_seats, x='object_type', y='number', ci=95, color='#197C78', ax=ax)
ax.set(title='Average seats by establishment type (95% confidence interval)', xlabel='', ylabel='Seats per venue')
ax.tick_params(axis='x', rotation=20); sns.despine(); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(10,5.5))
sns.boxplot(data=valid_seats, x='object_type', y='number', showfliers=False, color='#BBD8D4', ax=ax)
ax.set(title='Seating capacity distribution by establishment type', xlabel='', ylabel='Seats (outliers hidden for readability)')
ax.tick_params(axis='x', rotation=20); sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
chain_seats = classified[classified.number.notna() & classified.number.gt(0)]
chain_capacity = chain_seats.groupby('chain').number.agg(venues='count', mean='mean', median='median', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75)).round(1)
chain_capacity.index = chain_capacity.index.map({False:'Independent',True:'Chain'})
display(chain_capacity)
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=chain_seats, x='chain', y='number', showfliers=False, color='#BBD8D4', ax=ax)
ax.set_xticks([0,1],['Independent','Chain']); ax.set(title='Seat capacity by chain status', xlabel='', ylabel='Seats (outliers hidden)')
sns.despine(); plt.tight_layout(); plt.show()
display(Markdown('**Interpretation.** Compare both location counts and the median/IQR: a chain with many small units points toward repeatable compact formats; a higher chain median points toward a larger standard footprint. The data describe existing venues only and cannot establish that either footprint is more profitable.'))

## 4. Café-chain comparables and seat-capacity test ☕

Confirmed comparables are venues classified as **Chain = True** and **type = Café**. Chain-coded venues in other categories whose names include coffee-related terms are listed separately for manual review; a name match alone does not confirm a café competitor. The observed range and middle 50% are reference points, not recommended limits.


In [ ]:
# Separate confirmed café-chain venues from name-based leads in other categories.
chain_true = valid_seats['chain'].eq(True).fillna(False)
cafe_type = valid_seats['object_type'].eq('Cafe')
cafe_network_mask = chain_true & cafe_type
name_type = valid_seats['object_name'].fillna('') + ' ' + valid_seats['object_type'].fillna('')
coffee_term_pattern = r'\b(?:CAFE|CAFETERIA|COFFEE|ESPRESSO|TEA|BOBA|MATCHA|LATTE|ROASTERS?)\b'
coffee_name_signal = name_type.str.contains(coffee_term_pattern, case=False, regex=True, na=False)
name_review_mask = chain_true & ~cafe_type & coffee_name_signal

cafe_name_rows = valid_seats[cafe_network_mask | name_review_mask].copy()
cafe_name_rows['match_basis'] = np.where(cafe_name_rows['object_type'].eq('Cafe'),
    'Confirmed type: Cafe', 'Name/type lead: review category')
cafe_name_table = (cafe_name_rows.groupby(['match_basis','object_type','object_name'])['number']
    .agg(venues='count', seats_min='min', seats_median='median', seats_max='max')
    .reset_index().sort_values(['venues','object_name'], ascending=[False,True])
    .set_index(['match_basis','object_type','object_name']).round(1))
print('Top 30 exact recorded names; spelling variants are not merged.')
print(f'Confirmed café-chain venues: {int(cafe_network_mask.sum()):,}; distinct exact names: {valid_seats.loc[cafe_network_mask, "object_name"].nunique():,}.')
print(f'Additional chain-coded name/type leads outside Café: {int(name_review_mask.sum()):,}.')
display(cafe_name_table.head(30))

cafe_network = valid_seats.loc[cafe_network_mask].copy()
cafe_seat_min = int(cafe_network['number'].min())
cafe_seat_max = int(cafe_network['number'].max())
cafe_q25 = float(cafe_network['number'].quantile(.25))
cafe_median = float(cafe_network['number'].median())
cafe_q75 = float(cafe_network['number'].quantile(.75))
display(pd.DataFrame([{'Venues':len(cafe_network),'Observed minimum':cafe_seat_min,
    'P25':cafe_q25,'Median':cafe_median,'P75':cafe_q75,'Observed maximum':cafe_seat_max}]).round(1))
display(Markdown(f'**Decision read.** Confirmed café-chain venues range from {cafe_seat_min} to {cafe_seat_max} seats; the middle half spans {cafe_q25:.0f}–{cafe_q75:.0f}, with a median of {cafe_median:.0f}. The extremes are observed values, not commercial requirements. Use the central range to frame test scenarios, then validate local demand and economics.'))

# Compare seat capacity against membership in the confirmed café-chain cohort.
comparison = valid_seats.copy()
comparison['is_cafe_network'] = cafe_network_mask.fillna(False).astype(bool)
comparison['is_name_review'] = name_review_mask.fillna(False).astype(bool)
within_central_range = (comparison['number'].ge(cafe_q25) & comparison['number'].le(cafe_q75))
comparison['comparison_group'] = 'Other venue — outside central range'
comparison.loc[within_central_range.fillna(False), 'comparison_group'] = 'Other venue — comparable capacity'
comparison.loc[comparison['is_name_review'], 'comparison_group'] = 'Name suggests café — review type'
comparison.loc[comparison['is_cafe_network'], 'comparison_group'] = 'Observed café chain'
display(comparison.groupby('comparison_group')['number'].agg(venues='count', median='median', minimum='min', maximum='max').sort_values('venues',ascending=False).round(1))
seat_network_rho = comparison['number'].corr(comparison['is_cafe_network'].astype(int), method='spearman')
other_places = comparison.loc[~comparison['is_cafe_network']]
share_other_in_central = 100 * (other_places['number'].ge(cafe_q25) & other_places['number'].le(cafe_q75)).mean()

# Fixed-seed vertical offsets keep overlapping observations visible and repeatable.
import plotly.express as px
type_order = ['Restaurant','Fast Food','Cafe','Pizza','Bar','Bakery']
type_position = {name:position for position,name in enumerate(type_order)}
comparison['type_position'] = comparison['object_type'].map(type_position).astype(float)
comparison['plot_y'] = comparison['type_position'] + np.random.RandomState(42).uniform(-.22,.22,size=len(comparison))
comparison['chain_display'] = comparison['chain'].astype(object).map({True:'Chain',False:'Independent'}).fillna('Unknown').astype(str)
fig = px.scatter(comparison, x='number', y='plot_y', color='comparison_group', hover_name='object_name',
    hover_data={'number':':.0f','object_type':True,'chain_display':True,'comparison_group':True,'plot_y':False,'type_position':False,'is_cafe_network':False,'is_name_review':False},
    color_discrete_map={'Observed café chain':'#197C78','Name suggests café — review type':'#D98841','Other venue — comparable capacity':'#6A9FB5','Other venue — outside central range':'#C7CDD1'},
    title='Seat capacity by venue type: café-chain comparables and other venues',
    labels={'number':'Seats per venue','plot_y':'Venue type position','comparison_group':'Comparison group'})
fig.add_vrect(x0=cafe_q25,x1=cafe_q75,fillcolor='#F2C14E',opacity=.14,line_width=0,annotation_text='Central 50% of café chains')
fig.update_yaxes(tickmode='array',tickvals=list(type_position.values()),ticktext=type_order,title='Venue type')
fig.update_layout(template='plotly_white',height=600,legend_title_text='Comparison group')
fig.show()
strength = 'very weak' if abs(seat_network_rho)<.10 else ('weak' if abs(seat_network_rho)<.30 else 'moderate or strong')
display(Markdown(f'**Decision read.** {share_other_in_central:.1f}% of other venues fall within the central café-chain range. Seat count has a {strength} association with confirmed cohort membership (Spearman ρ = {seat_network_rho:.3f}); capacity alone is a poor classifier. Use the range to frame options, not to infer demand or profitability. Name-based leads outside the café category require manual verification.'))

## Simpson’s paradox check: composition changes the comparison

We compare mean listed seats at chain and independent venues overall, then within each establishment type. Type is a plausible composition factor because restaurants, cafés and fast-food venues have very different seat distributions. A classic Simpson reversal requires the aggregate direction to reverse consistently within comparable strata; mixed within-type effects should be reported as heterogeneity, not forced into that label.


In [ ]:
# Compare chain and independent seat means overall and within each venue type.
simpson_data = valid_seats[valid_seats['chain'].notna()].copy()
seat_by_type = (simpson_data.groupby(['object_type', 'chain'])['number']
    .agg(venues='count', mean_seats='mean', median_seats='median').unstack('chain'))
seat_by_type.columns = [f'{metric}_{"Chain" if status else "Independent"}'
                        for metric, status in seat_by_type.columns]
seat_by_type['mean_difference_chain_minus_independent'] = (
    seat_by_type['mean_seats_Chain'] - seat_by_type['mean_seats_Independent'])
comparable_types = seat_by_type.dropna(subset=['mean_seats_Chain', 'mean_seats_Independent']).copy()

all_means = simpson_data.groupby('chain')['number'].mean()
all_difference = float(all_means.loc[True] - all_means.loc[False])
shared = simpson_data[simpson_data['object_type'].isin(comparable_types.index)]
shared_means = shared.groupby('chain')['number'].mean()
shared_difference = float(shared_means.loc[True] - shared_means.loc[False])
equal_type_difference = float(comparable_types['mean_difference_chain_minus_independent'].mean())
within_differences = comparable_types['mean_difference_chain_minus_independent']
strict_reversal = bool((within_differences.gt(0).all() and all_difference < 0) or
                       (within_differences.lt(0).all() and all_difference > 0))

simpson_table = comparable_types[['venues_Independent', 'venues_Chain',
    'mean_seats_Independent', 'mean_seats_Chain',
    'mean_difference_chain_minus_independent']].copy()
simpson_table = simpson_table.rename(columns={
    'venues_Independent':'Independent venues', 'venues_Chain':'Chain venues',
    'mean_seats_Independent':'Independent mean seats', 'mean_seats_Chain':'Chain mean seats',
    'mean_difference_chain_minus_independent':'Chain minus independent'})
display(simpson_table.round(1))
print(f'All types pooled: chain {all_means.loc[True]:.2f}, independent {all_means.loc[False]:.2f}; difference {all_difference:+.2f} seats.')
print(f'Common-support pooled comparison ({len(comparable_types)} types): difference {shared_difference:+.2f} seats.')
print(f'Equal-weight type-standardized difference: {equal_type_difference:+.2f} seats (sensitivity check, not a preferred market mix).')

# Paired means show how the direction varies by type; bakeries have no independent comparator.
plot_data = comparable_types.sort_values('mean_seats_Independent')
fig, ax = plt.subplots(figsize=(9, 5.2))
for y, (venue_type, row) in enumerate(plot_data.iterrows()):
    x_ind, x_chain = row['mean_seats_Independent'], row['mean_seats_Chain']
    ax.plot([x_ind, x_chain], [y, y], color='#BECBC8', linewidth=2, zorder=1)
    ax.scatter(x_ind, y, color='#197C78', s=72, zorder=2)
    ax.scatter(x_chain, y, color='#D98841', s=72, zorder=2)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data.index)
ax.set(title='Mean listed seats by type and chain status', xlabel='Mean seats per venue', ylabel='')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],marker='o',color='w',label='Independent',markerfacecolor='#197C78',markersize=8),
                   Line2D([0],[0],marker='o',color='w',label='Chain',markerfacecolor='#D98841',markersize=8)],
          title='Venue status', loc='lower right')
sns.despine(); plt.tight_layout(); plt.show()

within_signs = int(within_differences.gt(0).sum()), int(within_differences.lt(0).sum())
result = ('A strict Simpson reversal is present.' if strict_reversal else
          'A strict Simpson reversal is not present: within-type differences have mixed directions.')
display(Markdown(f'**Decision read.** {result} The pooled chain–independent gap is {all_difference:+.1f} seats, while the equal-type-weight sensitivity is {equal_type_difference:+.1f}, effectively zero. Chains average more seats in {within_signs[0]} comparable types and fewer in {within_signs[1]}; bakery is excluded because this dataset has no independent bakery comparator. The result is composition-sensitive and heterogeneous, so the pooled average cannot support a general claim that chains need fewer seats or perform better.'))

## 5. Street-level distribution

Street names are extracted by removing the leading street number and common unit/suite suffix. This is a practical parsing rule for ranking locations, not a geocoded street identity. Similar street names may remain split (for example, direction or spelling variants), so rankings are indicative.

In [ ]:
UNIT_RE = re.compile(r'\s+(?:#|APT\b|STE\b|SUITE\b|UNIT\b).*$', re.I)
def extract_street(address):
    if pd.isna(address): return pd.NA
    text = UNIT_RE.sub('', str(address).upper().strip())
    # Most source addresses begin with a street number; preserve directional prefixes.
    text = re.sub(r'^\s*\d+[A-Z]?\s+', '', text)
    return text.strip(' ,') or pd.NA

valid_type['street'] = valid_type.address.map(extract_street).astype('string')
street_counts = valid_type.dropna(subset=['street']).groupby('street').size().sort_values(ascending=False)
top10_streets = street_counts.head(10).rename('venues').reset_index()
single_restaurant_streets = int((street_counts == 1).sum())
print(f'Streets with exactly one venue: {single_restaurant_streets:,}')
display(top10_streets)
fig, ax = plt.subplots(figsize=(10,5.5))
sns.barplot(data=top10_streets, y='street', x='venues', color='#197C78', ax=ax)
ax.set(title='Ten streets with the most listed venues', xlabel='Number of venues', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()

In [ ]:
busy_streets = street_counts.head(15).index
busy = valid_type[valid_type.street.isin(busy_streets) & valid_type.number.notna() & valid_type.number.gt(0)]
fig, ax = plt.subplots(figsize=(12,6))
sns.boxplot(data=busy, x='street', y='number', showfliers=False, color='#BBD8D4', ax=ax)
ax.set(title='Seating capacity on the 15 streets with the most venues', xlabel='', ylabel='Seats per venue')
ax.tick_params(axis='x', rotation=55); sns.despine(); plt.tight_layout(); plt.show()
street_capacity = busy.groupby('street').number.agg(venues='count', median='median', mean='mean', q25=lambda s:s.quantile(.25), q75=lambda s:s.quantile(.75)).sort_values('venues',ascending=False).round(1)
display(street_capacity.head(15))
display(Markdown('**Interpretation.** Dense streets contain a range of capacities, so street popularity alone does not indicate the right footprint. Compare medians and the IQR across streets; differences may reflect land costs, format, or local mix, none of which are measured in this file. One-venue streets represent dispersed coverage, not proven white space.'))

## Search-interest diligence: Yandex Wordstat

**Access status:** the public Wordstat page explains the service, but query-level results require a signed-in account. No Los Angeles search volumes were visible in this review, so this report does not claim a measured search-demand estimate. The official guide describes monthly query dynamics, regional/city breakdowns and related queries; the API also requires an OAuth token. [Yandex Wordstat interface guide](https://yandex.com/support2/wordstat/en/interface/new) · [API access requirements](https://yandex.com/support2/wordstat/en/content/api-wordstat).

**Query set for the next validation pass:** “coffee shop Los Angeles”, “cafe Los Angeles”, “robot cafe Los Angeles” and “robot restaurant Los Angeles”. Where available, select Los Angeles/California, review monthly dynamics over the last two years and inspect related top queries. Record the date, exact phrase, selected geography, count, query share and affinity index.

Treat Wordstat as a Yandex-search signal only. It is not total U.S. search demand, foot traffic, willingness to pay or proof of a viable café; triangulate with local customer interviews, corridor traffic, competitor visits and a pilot. Results remain pending an authenticated export or screenshots.


## 6. Investor implications and capacity scenarios

### What the dataset supports
- Los Angeles has a large, varied restaurant base and a material chain segment, showing that repeatable multi-location operations exist in this market.
- Cafés, fast food, restaurants, bars, bakeries, and pizza venues coexist; the concept should compete on a clear customer occasion and reliable execution.
- Seating varies materially across types and streets. A single citywide average is not a site plan.

### What it does not support
This inventory has no revenue, rent, labor, transactions, traffic, opening date, robot uptime, customer satisfaction, or robot cost data. It cannot prove robot adoption will persist after novelty fades, quantify unit economics, select a neighborhood, or determine a final seat count.

In [ ]:
# Scenario framework rather than a final seat-count recommendation.
scenario_table = pd.DataFrame([
    {'Scenario':'Compact pilot','Illustrative capacity':'Below local café median','Customer service choice':'Robot-led service, with human help available','Purpose':'Test robot workflow, service quality, and peak throughput with lower fixed exposure','Decision gate':'Expand only if repeat visits, utilization, and labor savings meet pilot targets'},
    {'Scenario':'Mid-sized pilot','Illustrative capacity':'Around local café median','Customer service choice':'Hybrid service: customer can choose robot or staff assistance','Purpose':'Balance seating flexibility with manageable operating complexity','Decision gate':'Validate covers per seat, queueing, and robot-human handoffs'},
    {'Scenario':'Larger format','Illustrative capacity':'Above local café median','Customer service choice':'Customer chooses robot-led, hybrid, or staff-led ordering/service','Purpose':'Test group occasions and higher peak capacity','Decision gate':'Proceed only with site-specific demand evidence and robust robot reliability'}
])
display(scenario_table)
display(Markdown('**Recommendation.** Treat these as testable formats, not a precise seat recommendation. Use the local café median and quartiles from the table above as reference points once street and site are chosen. Preserve customer choice in each scenario: allow a low-friction human fallback in the compact pilot, a clear robot-versus-staff choice in the hybrid pilot, and all three service modes in a larger test. Compare uptake, satisfaction, accessibility, weekday/weekend and peak/off-peak seat utilization, repeat rate, service time, robot intervention rate, labor cost per cover, and contribution margin. Improve each option over time using customer feedback and operating results. A chain strategy is plausible only after the pilot demonstrates consistent unit economics and an operating playbook that works without novelty-driven demand.'))

## 7. Investor conclusion and next steps

### Decision: fund validation, not immediate expansion
The supply data support a **conditional pilot**, not an immediate commitment to a café network. The confirmed café-chain cohort has a 22-seat median and a 12–37-seat central range, but seat capacity alone barely distinguishes those venues from other establishments. The observed one-seat minimum and 143-seat maximum are not commercial targets.

Select a corridor only after checking foot traffic, rent and competitors. Test compact, midpoint and larger formats while allowing customers to choose robot-led, hybrid or staff-led service. Before opening, set thresholds for repeat visits after launch promotions, seat utilization by daypart, reliability, satisfaction and contribution margin. Release expansion capital only if results persist through ordinary weeks and the operating model works without launch novelty.

### Investor diligence before committing further capital
1. Compare candidate corridors by foot traffic, rent, competition and nearby office, residential and transit demand.
2. Define seat-utilization, queueing and repeat-visit thresholds before testing the three capacity scenarios.
3. Measure preference, satisfaction and accessibility across robot, hybrid and human service modes.
4. Track downtime, interventions, maintenance and robot operating cost, including human fallback coverage.
5. Build unit economics across rent, labor, food, capital expenditure, maintenance and financing; expand only when margin and service quality hold.
